# L3：用语义化工具记忆扩展智能体的工具使用规模

<div style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> <p>⏳ <b>注意 <code>（数据库启动中）</code>：</b>本 notebook 大约需要 30-60 秒才能就绪。等待期间可以先开始观看视频。</p>
<p>如果运行第一个单元格后看到 <tt>Admin connection failed</tt>，只需稍等片刻再重新运行——这不是凭证问题。</p>
</div>

### 工具的可扩展性问题

随着 AI 系统的成长，你可能会有**数百个工具**可用——API、数据库查询、计算器、搜索引擎等等。然而，在推理时把所有工具都传给 LLM 会带来严重问题：

| 问题 | 影响 |
|---------|--------|
| **上下文膨胀** | 工具定义消耗 token，留给实际内容的空间变少 |
| **工具选择失败** | 面对过多选项时，LLM 很难选对工具 |
| **延迟增加** | token 越多 = 推理越慢 |
| **成本更高** | token 越多 = API 费用越高 |

OpenAI、Anthropic 等模型厂商通常建议限制暴露给 LLM 的工具数量（要保证选择可靠，一般最多 10-20 个）。

### 解决方案：语义化工具检索（Semantic Tool Retrieval）

`Toolbox` 类把工具当作一种**可检索的记忆**来解决这个问题：

1. **注册数百个工具** —— 把所有可用工具连同描述和嵌入向量一起存储
2. **只检索相关工具** —— 推理时用向量检索找出与当前查询语义相关的工具
3. **传入聚焦的工具集** —— 只把检索到的工具（通常 3-5 个）传给 LLM

这种方式意味着你的系统可以**扩展到数百个工具**，而 LLM 每次只看到与当前查询最相关的那几个。

### 代码如何工作

`Toolbox` 类用 **docstring 作为检索键**：

```
用户查询 → 嵌入查询 → 向量检索 → 找到 docstring 语义相近的工具 → 返回相关工具
```

| 组件 | 用途 |
|-----------|---------|
| `Toolbox`（来自 `helper.py`） | 各课共用的类，负责注册和检索工具 |
| `ToolMetadata`（在 `helper.py` 内） | 存储工具名称、描述、签名、参数 |
| `_augment_docstring()` | 用 LLM 改写 docstring，提升检索效果 |
| `_generate_queries()` | 生成会触发该工具的合成查询（synthetic queries） |
| `register_tool()` | 装饰器，把工具连同其嵌入向量存入工具箱 |

当你调用 `memory_manager.read_toolbox(query)` 时，它会做一次相似度检索，找出 docstring 与查询语义相近的工具。

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>访问 <code>requirements.txt</code> 和 <code>helper.py</code> 文件：</b> 1) 点击 notebook 顶部菜单中的 <em>"File"</em>，然后 2) 点击 <em>"Open"</em>。

<p> ⬇ &nbsp; <b>下载 Notebook：</b> 1) 点击 notebook 顶部菜单中的 <em>"File"</em>，然后 2) 点击 <em>"Download as"</em> 并选择 <em>"Notebook (.ipynb)"</em>。</p>

</div>

## Part 0：连接数据库

### 第 1 步：建立一个可用的数据库会话

下一个单元格会检查 Docker、按需启动 Oracle，并为向量操作准备好数据库用户/schema。
可以把这一步看作在任何智能体记忆逻辑运行之前的基础设施引导（bootstrapping）。接下来我们打开一个所有记忆组件都会共享的连接对象。
这个连接是 SQL 记忆（对话历史）和向量记忆存储共同的骨干。

In [ ]:
from helper import suppress_warnings

# 警告控制
suppress_warnings()

from helper import load_env, setup_oracle_database, connect_to_oracle

load_env()

# 一次性管理员初始化：配置表空间、向量内存和 VECTOR 用户
setup_oracle_database()

# 后续所有操作都以 VECTOR 用户身份连接
database_connection = connect_to_oracle(
    user="VECTOR",
    password="VectorPwd_2025",
    dsn="127.0.0.1:1521/FREEPDB1",
    program="devrel.deeplearning.course_1",
)

print("Using user:", database_connection.username)

## Part 1：加载嵌入模型

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
# 初始化嵌入模型
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-mpnet-base-v2"
)

### 为什么嵌入对工具使用很重要

在接下来的步骤中，`Toolbox` 用嵌入向量把自然语言查询映射到最相关的工具。
这意味着工具检索是**语义化**的：即使查询的措辞和工具名并不完全一致，智能体也能发现可用的能力。

### 加载模型、StoreManager 和 MemoryManager

### 引入 LLM 运行时

在增强工具元数据或运行智能体决策之前，先初始化 OpenAI 客户端——工具箱和后面的 agent loop 都会用到它。

In [ ]:
from openai import OpenAI

client = OpenAI()

### 定义记忆存储的名称

下一个单元格为每种记忆类型统一表名。
给每个存储显式命名，能让 AI 系统开发者更容易调试，也更容易保持课与课之间的连续性。

In [ ]:
# 每种记忆类型对应的表名
CONVERSATIONAL_TABLE = "CONVERSATIONAL_MEMORY"
KNOWLEDGE_BASE_TABLE = "SEMANTIC_MEMORY"
WORKFLOW_TABLE = "WORKFLOW_MEMORY"
TOOLBOX_TABLE = "TOOLBOX_MEMORY"
ENTITY_TABLE = "ENTITY_MEMORY"
SUMMARY_TABLE = "SUMMARY_MEMORY"
TOOL_LOG_TABLE = "TOOL_LOG_MEMORY"

### 从头开始：删除已有的表

<p style="background-color:#ff9a94; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>注意：</b>为了保证无论前面几课是否运行过，本课都能正确运行，我们先运行下面的单元格删除所有记忆表再重建。这样能保证一个干净的初始状态：距离策略一致、没有残留的旧数据。</p>


In [ ]:
ALL_TABLES = [
    CONVERSATIONAL_TABLE,
    KNOWLEDGE_BASE_TABLE,
    WORKFLOW_TABLE,
    TOOLBOX_TABLE,
    ENTITY_TABLE,
    SUMMARY_TABLE,
    TOOL_LOG_TABLE]

# 删除已存在的表，从头开始
for table in ALL_TABLES:
    try:
        with database_connection.cursor() as cur:
            cur.execute(f"DROP TABLE {table} PURGE")
            print(f"  - {table} (dropped)")
    except Exception as e:
        if "ORA-00942" in str(e):
            print(f"  - {table} (not exists)")
        else:
            print(f"  ✗ {table}: {e}")

database_connection.commit()

### 准备对话表

这一步确保对话记忆的 SQL 表存在。
对话记忆是你的短程线程历史，是对话连续性的锚点。

In [ ]:
# 创建或获取对话历史表
from helper import create_conversational_history_table, create_tool_log_table

CONVERSATION_HISTORY_TABLE = create_conversational_history_table(database_connection, CONVERSATIONAL_TABLE)
TOOL_LOG_HISTORY_TABLE = create_tool_log_table(database_connection, TOOL_LOG_TABLE)


### 通过 `StoreManager` 搭建向量存储层

下面的单元格把每个向量记忆存储（知识库、工作流、工具箱、实体、摘要）都接到一个统一的 manager 上。
这样你得到的是干净的 getter 方法，而不是散落各处的初始化逻辑。

In [ ]:
from langchain_oracledb.vectorstores import OracleVS
from langchain_community.vectorstores.utils import DistanceStrategy
from helper import StoreManager

# 创建 StoreManager 实例
store_manager = StoreManager(
    client=database_connection,
    embedding_function=embedding_model,
    table_names={
        'knowledge_base': KNOWLEDGE_BASE_TABLE,
        'workflow': WORKFLOW_TABLE,
        'toolbox': TOOLBOX_TABLE,
        'entity': ENTITY_TABLE,
        'summary': SUMMARY_TABLE,
    },
    distance_strategy=DistanceStrategy.COSINE,
    conversational_table=CONVERSATION_HISTORY_TABLE,
    tool_log_table=TOOL_LOG_HISTORY_TABLE,
)

# 通过 manager 获取所有存储
conversation_table = store_manager.get_conversational_table()
knowledge_base_vs = store_manager.get_knowledge_base_store()
workflow_vs = store_manager.get_workflow_store()
toolbox_vs = store_manager.get_toolbox_store()
entity_vs = store_manager.get_entity_store()
summary_vs = store_manager.get_summary_store()
tool_log_table = store_manager.get_tool_log_table()

print("✅ All stores loaded via StoreManager")

### 初始化记忆编排 + Toolbox 实例

现在我们组装运行时：`MemoryManager` 统一了对所有记忆存储的读/写访问，`Toolbox` 负责语义化地注册/检索工具。
这里我们仍然初始化一个具体的 `toolbox` 实例，以便在本 notebook 中注册工具。

In [ ]:
from helper import MemoryManager, Toolbox

# 初始化 MemoryManager 实例
memory_manager = MemoryManager(
    conn=database_connection,
    conversation_table=conversation_table,
    knowledge_base_vs=knowledge_base_vs,
    workflow_vs=workflow_vs,
    toolbox_vs=toolbox_vs,
    entity_vs=entity_vs,
    summary_vs=summary_vs,
    tool_log_table=TOOL_LOG_HISTORY_TABLE
)

# 初始化 Toolbox
toolbox = Toolbox(memory_manager, client, embedding_model)

print("✅ MemoryManager and Toolbox initialized")

## 工具总览

本课将创建以下工具并注册到 Toolbox：

| 工具 | 用途 |
|------|---------|
| `search_tavily` | 用 Tavily API 搜索网页，并把结果持久化到知识库供未来检索 |
| `get_current_time` | 返回当前日期和时间（可选带微秒的详细格式） |
| `arxiv_search_candidates` | 在 arXiv 上按查询搜索论文，返回元数据（ID、标题、作者、摘要） |
| `fetch_and_save_paper_to_kb_db` | 下载 arXiv 论文 PDF，抽取文本、切块（chunk），存入知识库 |

每个工具都通过 `@toolbox.register_tool()` 注册，注册时会存储其嵌入向量用于语义检索。当智能体收到查询时，只有最相关的工具会被检索出来传给 LLM。

>我们把工具箱检索同时暴露为**程序化操作**和**智能体可调用的技能**。这让智能体在执行途中需要初始提供之外的能力时，可以自主地查询工具。

In [ ]:
@toolbox.register_tool(augment=True)
def read_toolbox(query: str, k: int = 3) -> list[str]:
    """
    在工具箱中搜索能帮助解决问题或完成任务的函数。

    以下情况使用本工具：
    - 遇到错误或非预期输出，需要换一种方法
    - 当前可用的工具似乎不足以完成任务
    - 需要发现针对某个特定问题有哪些可用能力
    - 想找到可能更好处理边界情况的替代函数

    参数:
        query: 用自然语言描述你想完成的事情或想解决的问题。
               对任务或所遇错误描述得越具体，结果越好。
        k: 返回的相关工具数量（默认: 5）

    返回:
        与查询语义匹配的工具定义列表，
        包括工具名称、描述和参数 schema。

    示例查询:
        - "search for academic papers on machine learning"
        - "fetch and store document content"
        - "get the current date and time"
        - "summarize long text and save to memory"
    """
    return memory_manager.read_toolbox(query, k=k)

## Part 2：用 Tavily 访问网页

本节演示如何创建一个 LLM 可调用的**智能体工具（agentic tool）**来搜索网页。

我们使用 [Tavily](https://tavily.com/)——一个为 LLM 应用设计的 AI 优化搜索 API。

本节做什么

1. **初始化 Tavily 客户端** —— 用 API key 配置搜索 API
2. **把 `search_tavily` 注册为工具** —— 用 `@toolbox.register_tool(augment=True)` 使其可被发现
3. **实现"搜索即存储"模式** —— 结果自动写入知识库记忆
4. **测试工具检索** —— 验证该工具能通过语义检索被找到

### "搜索即存储"模式（Search-and-Store）

值得注意的是：我们不仅获得了智能体执行时原本拿不到的外部上下文，还把它持久化到知识库记忆中，智能体在后续迭代中可以复用这些信息。
当智能体调用 `search_tavily()` 时，它不只是返回结果——还会**把结果持久化到知识库**：

```
智能体调用 search_tavily("latest AI news")
       ↓
Tavily API 返回结果
       ↓
每条结果连同 metadata（标题、URL、时间戳）写入 knowledge_base_vs
       ↓
未来的查询可以直接检索到这些信息，无需再次搜索
```

这个模式意味着智能体能从它的搜索中**学习**。一次发现的信息会成为智能体长期记忆的一部分，未来对话可直接使用，不需要额外的 API 调用。

In [ ]:
from tavily import TavilyClient
from datetime import datetime

tavily_client = TavilyClient()

@toolbox.register_tool(augment=True)
def search_tavily(query: str, max_results: int = 5):
    """
    用这个函数搜索网页，并把结果存入知识库。
    """
    response = tavily_client.search(query=query, max_results=max_results)
    results = response.get("results", [])

    # 把每条结果写入知识库
    for result in results:
        # 构造用于嵌入的文本内容
        text = f"Title: {result.get('title', '')}\nContent: {result.get('content', '')}\nURL: {result.get('url', '')}"

        # 构造 metadata
        metadata = {
            "title": result.get("title", ""),
            "url": result.get("url", ""),
            "score": result.get("score", 0),
            "source_type": "tavily_search",
            "query": query,
            "timestamp": datetime.now().isoformat()
        }

        # 写入知识库
        memory_manager.write_knowledge_base(text, metadata)

    return results

### 增强版 vs 原始版 Docstring

当 `augment=True` 时，`Toolbox` 会把**原始 docstring** 和**函数源代码**一起发给 LLM，由 LLM 产出一段更丰富、更详细的描述。被嵌入和存储的正是这段增强后的文本——从而提升语义可分性和检索召回率。

我们来对比一下 `search_tavily` 的**原始**一行 docstring 和 LLM 通过分析代码产出的**增强**版本：

In [ ]:
import inspect

# 原始 docstring（开发者写的——只有一行）
original = ("Use this function to search the web"
            " and store the results in the"
            " knowledge base.")

# 获取函数的实际源代码
fn = toolbox._tools_by_name["search_tavily"]
source = inspect.getsource(fn)

print("ORIGINAL DOCSTRING:")
print(f'  "{original}"')
print()

# LLM 会同时读取 docstring 和源代码
augmented = toolbox._augment_docstring(original, source)

print("AUGMENTED DOCSTRING (LLM-enhanced):")
print(f"  {augmented}")

### 先加一个简单的工具函数

在注册高级检索工具之前，先注册一个确定性的小工具（`get_current_time`）。
这是个不错的教学模式：先在低风险的函数上验证工具注册流程。

In [ ]:
from datetime import datetime

@toolbox.register_tool(augment=True)
def get_current_time(detailed: bool = False) -> str:
    """
    返回当前时间。

    参数:
        detailed: 为 True 时返回带微秒的详细格式

    返回:
        str: 格式化后的当前时间字符串
    """
    if detailed:
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")
    else:
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

### 配置 arXiv 候选论文检索

下一个单元格搭建一个轻量的候选论文检索器（只取标题、摘要、元数据，不取整份 PDF）。
这样可以在昂贵的全文灌入之前先做快速探索。

In [ ]:
from langchain_community.retrievers import ArxivRetriever

arxiv_retriever = ArxivRetriever(
    load_max_docs=8,
    get_full_documents=False,
    doc_content_chars_max=4000
)

### 注册一个 arXiv 发现工具

本节添加 `arxiv_search_candidates`，它返回结构化的 JSON 候选列表，智能体可以先在这些候选上推理，再决定要灌入哪篇论文。

In [ ]:
import json
from urllib.parse import urlparse

def _arxiv_id_from_entry_id(entry_id: str) -> str:
    """
    把 'http://arxiv.org/abs/2310.08560v2' 转换成 '2310.08560v2'
    """
    if not entry_id:
        return ""
    path = urlparse(entry_id).path  # 例如 '/abs/2310.08560v2'
    return path.split("/abs/")[-1].strip("/")

@toolbox.register_tool(augment=False)
def arxiv_search_candidates(query: str, k: int = 5) -> str:
    """
    搜索 arXiv，返回候选论文的 JSON 列表（含 ID + 元数据）。

    输出 schema（JSON 字符串）:
    [
      {
        "arxiv_id": "2310.08560v2",
        "entry_id": "http://arxiv.org/abs/2310.08560v2",
        "title": "...",
        "authors": "...",
        "published": "2024-02-12",
        "abstract": "..."
      },
      ...
    ]
    """
    docs = arxiv_retriever.invoke(query)
    candidates = []
    for d in (docs or [])[:k]:
        meta = d.metadata or {}
        entry_id = meta.get("Entry ID", "")
        candidates.append({
            "arxiv_id": _arxiv_id_from_entry_id(entry_id),
            "entry_id": entry_id,
            "title": meta.get("Title", ""),
            "authors": meta.get("Authors", ""),
            "published": str(meta.get("Published", "")),
            "abstract": (d.page_content or "")[:2500],
        })
    return json.dumps(candidates, ensure_ascii=False, indent=2)

### 注册深度灌入工具：抓取、切块、持久化

接下来定义一个更重的工具：下载论文全文、按嵌入模型的输入限制切块（chunk），并存入知识库记忆。
这展示了一个生产级模式：**把大负载的处理移出模型上下文，交给记忆基础设施**。

In [ ]:
from datetime import timezone
from langchain_community.document_loaders import ArxivLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


@toolbox.register_tool(augment=True)
def fetch_and_save_paper_to_kb_db(
    arxiv_id: str,
    chunk_size: int = 1500,
    chunk_overlap: int = 200,
) -> str:
    """
    抓取 arXiv 论文全文（PDF -> 文本），切块后存入 OracleVS
    知识库表（避免让全文内容经过 LLM 上下文中转）。

    """

    # 1) 从 arXiv 加载论文全文（PDF -> 文本）
    loader = ArxivLoader(
        query=arxiv_id,
        load_max_docs=1,
        doc_content_chars_max=None,  # 当前 LangChain 文档中表示"不截断"
    )
    docs = loader.load()
    if not docs:
        return f"No documents found for arXiv id: {arxiv_id}"

    doc = docs[0]

    title = (
        doc.metadata.get("Title")
        or doc.metadata.get("title")
        or f"arXiv {arxiv_id}"
    )

    # 归一化常见的 arxiv 元数据键名
    entry_id = doc.metadata.get("Entry ID") or doc.metadata.get("entry_id") or ""
    published = doc.metadata.get("Published") or doc.metadata.get("published") or ""
    authors = doc.metadata.get("Authors") or doc.metadata.get("authors") or ""

    full_text = doc.page_content or ""
    if not full_text.strip():
        return f"Loaded arXiv {arxiv_id} but extracted empty text (PDF parsing issue)."

    # 2) 切块（重要：嵌入模型有输入长度限制，切块可避免失败/截断）
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    chunks = splitter.split_text(full_text)

    # 3) 把块存入 OracleVS（向量存储表）
    ts_utc = datetime.now(timezone.utc).isoformat()
    metadatas = []
    for i in range(len(chunks)):
        metadatas.append(
            {
                "source": "arxiv",
                "arxiv_id": arxiv_id,
                "title": title,
                "entry_id": entry_id,
                "published": str(published),
                "authors": str(authors),
                "chunk_id": i,
                "num_chunks": len(chunks),
                "ingested_ts_utc": ts_utc,
            }
        )

    memory_manager.write_knowledge_base(chunks, metadatas)

    return (
        f"Saved arXiv {arxiv_id} to {KNOWLEDGE_BASE_TABLE}: "
        f"{len(chunks)} chunks (title: {title})."
    )


### 验证语义化工具检索

最后，我们对工具箱记忆发起一个自然语言查询，验证检索质量。
如果这一步能跑通，你的智能体就能在运行时动态收窄工具选择范围，而不是把所有工具都塞进 prompt 上下文。

In [ ]:
import pprint
retrieved_tools = memory_manager.read_toolbox("Get more details on a paper on AI", k=1)
pprint.pprint(retrieved_tools)